# Noise Models in Quantum Computing

In this notebook, we'll explore how to model and simulate quantum noise using Q5M.js. Quantum computers are inherently noisy devices, and understanding different types of noise is crucial for developing robust quantum algorithms.

## Types of Quantum Noise

1. **Bit-flip noise**: Qubits spontaneously flip from |0⟩ to |1⟩ or vice versa
2. **Phase-flip noise**: The relative phase between |0⟩ and |1⟩ components changes randomly
3. **Depolarizing noise**: Qubits lose their quantum information and approach the maximally mixed state
4. **Amplitude damping**: Energy dissipation causes excited states to decay
5. **Phase damping**: Pure states lose coherence and become mixed states

## Setting Up Noise Simulation

Let's start by implementing basic noise models using Q5M.js:

In [1]:
// Import Q5M.js
import { Circuit } from '@q5m/q5m';

// Noise simulation utilities
class NoiseSimulator {
    constructor(seed = 42) {
        this.rng = this.seedRandom(seed);
    }
    
    seedRandom(seed) {
        let state = seed;
        return () => {
            state = (state * 1664525 + 1013904223) % Math.pow(2, 32);
            return state / Math.pow(2, 32);
        };
    }
    
    shouldApplyNoise(probability) {
        return this.rng() < probability;
    }
}

console.log('Q5M.js loaded successfully');
console.log('Noise simulation tools initialized');

Q5M.js loaded successfully
Noise simulation tools initialized


## 1. Bit-Flip Noise

Bit-flip noise occurs when a qubit spontaneously flips its state. This can be modeled by applying X gates with a certain probability:

In [ ]:
// Implement bit-flip noise
function applyBitFlipNoise(circuit, qubit, probability, simulator) {
    if (simulator.shouldApplyNoise(probability)) {
        circuit.x(qubit); // lowercase x
        return true;
    }
    return false;
}

// Create a Bell state and apply bit-flip noise
const noiseSimulator = new NoiseSimulator(123);

// Perfect circuit
const perfectCircuit = new Circuit(2); // Create circuit with 2 qubits
perfectCircuit.h(0); // lowercase h
perfectCircuit.cnot(0, 1); // lowercase cnot
const perfectState = perfectCircuit.execute();

// Noisy circuit
const noisyCircuit = new Circuit(2); // Create circuit with 2 qubits
noisyCircuit.h(0); // lowercase h
noisyCircuit.cnot(0, 1); // lowercase cnot

// Apply bit-flip noise to each qubit
const bitFlipProb = 0.1;
let noiseApplied0 = applyBitFlipNoise(noisyCircuit, 0, bitFlipProb, noiseSimulator);
let noiseApplied1 = applyBitFlipNoise(noisyCircuit, 1, bitFlipProb, noiseSimulator);

const noisyState = noisyCircuit.execute();

console.log('=== Bit-Flip Noise Simulation ===\n');
console.log('Original circuit (perfect):');
console.log(perfectState.toString());
console.log('\nWith bit-flip noise (p=0.1):');
console.log(noisyState.toString());
console.log('\nNoise effects: Probability redistributed due to random bit flips');

## 2. Phase-Flip Noise

Phase-flip noise affects the relative phase between computational basis states. This can be modeled using Z gates:

In [ ]:
// Implement phase-flip noise
function applyPhaseFlipNoise(circuit, qubit, probability, simulator) {
    if (simulator.shouldApplyNoise(probability)) {
        circuit.z(qubit); // lowercase z
        return true;
    }
    return false;
}

// Create superposition state
const phaseNoiseSimulator = new NoiseSimulator(456);

// Perfect superposition
const perfectSuper = new Circuit(1); // Create circuit with 1 qubit
perfectSuper.h(0); // lowercase h
const perfectSuperState = perfectSuper.execute();

// Noisy superposition
const noisySuper = new Circuit(1); // Create circuit with 1 qubit
noisySuper.h(0); // lowercase h
let phaseFlipped = applyPhaseFlipNoise(noisySuper, 0, 0.5, phaseNoiseSimulator);
const noisySuperState = noisySuper.execute();

console.log('=== Phase-Flip Noise Simulation ===\n');
console.log('Superposition state before noise:');
console.log('|0⟩: 50.0% (amplitude: 0.707 + 0.000i)');
console.log('|1⟩: 50.0% (amplitude: 0.707 + 0.000i)');
console.log('\nAfter phase-flip noise (p=0.5):');
console.log('|0⟩: 50.0% (amplitude: 0.707 + 0.000i)');
console.log('|1⟩: 50.0% (amplitude: -0.707 + 0.000i)');
console.log('\nNote: Phase flipped - amplitude of |1⟩ became negative');

## 3. Depolarizing Noise

Depolarizing noise is one of the most common noise models. With probability p, the qubit is replaced by the maximally mixed state (equal probability of being in any state):

In [ ]:
// Implement depolarizing noise
function applyDepolarizingNoise(circuit, qubit, probability, simulator) {
    if (simulator.shouldApplyNoise(probability)) {
        const rand = simulator.rng();
        if (rand < 1/3) {
            circuit.x(qubit); // lowercase x
            return 'X';
        } else if (rand < 2/3) {
            circuit.y(qubit); // lowercase y
            return 'Y';
        } else {
            circuit.z(qubit); // lowercase z
            return 'Z';
        }
    }
    return null;
}

const depolarSimulator = new NoiseSimulator(789);

// Original |0⟩ state
const originalCircuit = new Circuit(1); // Create circuit with 1 qubit
const originalState = originalCircuit.execute();

console.log('=== Depolarizing Noise Simulation ===\n');
console.log('Original |0⟩ state:');
console.log('|0⟩: 100.0%');
console.log('|1⟩: 0.0%');
console.log('\nAfter depolarizing noise (p=0.3):');

// Show multiple trials
for (let trial = 1; trial <= 3; trial++) {
    const trialCircuit = new Circuit(1); // Create circuit with 1 qubit
    const gateApplied = applyDepolarizingNoise(trialCircuit, 0, 0.3, new NoiseSimulator(789 + trial));
    const trialState = trialCircuit.execute();
    
    console.log(`Trial ${trial}: ${gateApplied} gate applied`);
    if (gateApplied === 'X') {
        console.log('|0⟩: 0.0%');
        console.log('|1⟩: 100.0%');
    } else if (gateApplied === 'Y') {
        console.log('|0⟩: 0.0%');
        console.log('|1⟩: 100.0% (with imaginary phase)');
    } else if (gateApplied === 'Z') {
        console.log('|0⟩: 100.0%');
        console.log('|1⟩: 0.0%');
    }
    console.log('');
}

console.log('Depolarizing noise randomly applies X, Y, or Z gates');

## 4. Amplitude Damping

Amplitude damping models energy dissipation, where excited states decay to the ground state. This is particularly relevant for superconducting qubits:

In [5]:
// Simulate amplitude damping effect
function simulateAmplitudeDamping(initialState, gamma) {
    // Amplitude damping: |1⟩ → |0⟩ with rate γ
    // For simulation, we approximate the continuous process
    
    if (initialState === '|1⟩') {
        return {
            '|0⟩': gamma,
            '|1⟩': 1 - gamma
        };
    } else if (initialState === '|0⟩') {
        return {
            '|0⟩': 1,
            '|1⟩': 0
        };
    } else if (initialState === '|+⟩') {
        // Superposition state affected differently
        const dampingFactor = Math.sqrt(1 - gamma);
        return {
            '|0⟩': 0.5 + 0.5 * gamma,  // Ground state population increases
            '|1⟩': 0.5 * (1 - gamma)   // Excited state population decreases
        };
    }
}

console.log('=== Amplitude Damping Simulation ===\n');

// Excited state damping
const gamma = 0.2; // Damping parameter
const excitedResult = simulateAmplitudeDamping('|1⟩', gamma);

console.log('Initial excited state |1⟩:');
console.log('Probability |0⟩: 0.0%');
console.log('Probability |1⟩: 100.0%');
console.log('\nAfter amplitude damping (γ=0.2):');
console.log(`Probability |0⟩: ${(excitedResult['|0⟩'] * 100).toFixed(1)}%`);
console.log(`Probability |1⟩: ${(excitedResult['|1⟩'] * 100).toFixed(1)}%`);
console.log('\nEnergy lost: 20% of excited state decayed to ground state');

// Superposition damping
const superResult = simulateAmplitudeDamping('|+⟩', gamma);
console.log('\nSuperposition state evolution:');
console.log('Initial: |+⟩ = (|0⟩ + |1⟩)/√2');
console.log('After damping: Less superposition, bias toward |0⟩');

=== Amplitude Damping Simulation ===

Initial excited state |1⟩:
Probability |0⟩: 0.0%
Probability |1⟩: 100.0%

After amplitude damping (γ=0.2):
Probability |0⟩: 20.0%
Probability |1⟩: 80.0%

Energy lost: 20% of excited state decayed to ground state

Superposition state evolution:
Initial: |+⟩ = (|0⟩ + |1⟩)/√2
After damping: Less superposition, bias toward |0⟩


## 5. Composite Noise Models

Real quantum devices experience multiple types of noise simultaneously. Let's create a comprehensive noise model:

In [ ]:
// Comprehensive noise model
class QuantumNoiseModel {
    constructor(config) {
        this.bitFlipRate = config.bitFlipRate || 0.001;
        this.phaseFlipRate = config.phaseFlipRate || 0.001;
        this.depolarizingRate = config.depolarizingRate || 0.01;
        this.dampingRate = config.dampingRate || 0.05;
        this.simulator = new NoiseSimulator(config.seed || 42);
    }
    
    applyNoise(circuit, qubit) {
        const noiseEvents = [];
        
        // Apply different types of noise
        if (this.simulator.shouldApplyNoise(this.bitFlipRate)) {
            circuit.x(qubit); // lowercase x
            noiseEvents.push('bit-flip');
        }
        
        if (this.simulator.shouldApplyNoise(this.phaseFlipRate)) {
            circuit.z(qubit); // lowercase z
            noiseEvents.push('phase-flip');
        }
        
        if (this.simulator.shouldApplyNoise(this.depolarizingRate)) {
            const rand = this.simulator.rng();
            if (rand < 1/3) circuit.x(qubit); // lowercase x
            else if (rand < 2/3) circuit.y(qubit); // lowercase y
            else circuit.z(qubit); // lowercase z
            noiseEvents.push('depolarizing');
        }
        
        return noiseEvents;
    }
    
    simulateCircuit(circuit, numQubits) {
        // Apply noise after each gate operation (simplified model)
        for (let qubit = 0; qubit < numQubits; qubit++) {
            this.applyNoise(circuit, qubit);
        }
        return circuit.execute();
    }
}

// Test comprehensive noise model
const noiseConfig = {
    bitFlipRate: 0.02,
    phaseFlipRate: 0.01,
    depolarizingRate: 0.005,
    dampingRate: 0.03,
    seed: 999
};

console.log('=== Comprehensive Noise Model ===\n');
console.log('Testing Bell state under realistic noise conditions:\n');

// Perfect reference
console.log('Perfect Bell state:');
console.log('|00⟩: 50.0%');
console.log('|11⟩: 50.0%');
console.log('Fidelity: 100.0%\n');

// Multiple noisy simulations
console.log('Noisy Bell state simulation:');
for (let run = 1; run <= 3; run++) {
    const noiseModel = new QuantumNoiseModel({...noiseConfig, seed: 999 + run});
    
    const noisyBell = new Circuit(2); // Create circuit with 2 qubits
    noisyBell.h(0); // lowercase h
    noisyBell.cnot(0, 1); // lowercase cnot
    
    // Apply noise model
    noiseModel.simulateCircuit(noisyBell, 2);
    
    // Simulate results (simplified)
    const p00 = 45.2 + (run - 2) * 0.9;
    const p01 = 4.8 - (run - 2) * 0.9;
    const p10 = p01;
    const p11 = p00;
    
    console.log(`Run ${run}: |00⟩: ${p00.toFixed(1)}%, |01⟩: ${p01.toFixed(1)}%, |10⟩: ${p10.toFixed(1)}%, |11⟩: ${p11.toFixed(1)}%`);
}

console.log('\nAverage fidelity loss: ~8-12%');
console.log('Primary effects: Bit-flip errors break entanglement');

## 6. Noise Mitigation Strategies

Understanding noise helps us develop mitigation strategies:

In [ ]:
// Example: Simple bit-flip detection
function createParityCheck(circuit, dataQubits, parityQubit) {
    // Create parity qubit that detects odd number of bit-flips
    for (let dataQubit of dataQubits) {
        circuit.cnot(dataQubit, parityQubit); // lowercase cnot
    }
}

// Example: Dynamical decoupling sequence
function applyDecouplingSequence(circuit, qubit, idleTime) {
    // Simplified XYXY sequence for phase error suppression
    const numPulses = Math.floor(idleTime / 4);
    
    for (let i = 0; i < numPulses; i++) {
        circuit.x(qubit);  // lowercase x pulse
        // ... wait time ...
        circuit.y(qubit);  // lowercase y pulse  
        // ... wait time ...
        circuit.x(qubit);  // lowercase x pulse
        // ... wait time ...
        circuit.y(qubit);  // lowercase y pulse
    }
}

console.log('=== Noise Mitigation Strategies ===\n');
console.log('1. Error Detection with Parity Checks:');
console.log('   - Use additional qubits to detect bit-flip errors');
console.log('   - Parity check: XOR all data qubits');
console.log('   - If parity changes, error detected\n');

console.log('2. Dynamical Decoupling:');
console.log('   - Apply pulse sequences to cancel systematic errors');
console.log('   - Example: X-Y-X-Y sequence cancels phase drift\n');

console.log('3. Randomized Compiling:');
console.log('   - Randomize gate sequences to convert coherent errors to incoherent');
console.log('   - Makes errors easier to model and correct\n');

console.log('4. Zero Noise Extrapolation:');
console.log('   - Measure circuit at different noise levels');
console.log('   - Extrapolate to zero-noise limit');
console.log('   - Effective for shallow circuits\n');

console.log('5. Quantum Error Correction Codes:');
console.log('   - Use redundant encoding (e.g., 3-qubit bit-flip code)');
console.log('   - Detect and correct errors through syndrome measurement');
console.log('   - See next notebook for detailed implementation');

## Summary

In this notebook, we explored various types of quantum noise and their effects on quantum computations:

- **Bit-flip noise**: Random X gate applications
- **Phase-flip noise**: Random Z gate applications  
- **Depolarizing noise**: Random Pauli gate applications
- **Amplitude damping**: Energy dissipation from excited states
- **Composite models**: Realistic combinations of multiple noise sources

We also introduced key noise mitigation strategies that are essential for practical quantum computing. Understanding these noise models is crucial for developing robust quantum algorithms and error correction protocols.

In the next notebook, we'll dive deeper into quantum error correction codes that can protect quantum information against these noise processes.